# Fleet Scoring — ML Model Predictions

Loads trained models from the **Snowflake Model Registry** and scores the entire fleet using actual FEATURE_STORE data. Pure ML predictions — no hardcoded overrides.

**Prerequisites:**
- Run `ml_training` notebook first to train and register models
- Models `FAILURE_CLASSIFIER`, `RUL_REGRESSOR`, `PROBABILITY_CALIBRATOR` (v7) must exist in `PDM_DEMO.ML`
- `ML.MODEL_METADATA` table must contain baselines and model version

**Output:** `PDM_DEMO.ANALYTICS.PREDICTIONS` — ~3,500 daily predictions for 50 assets × ~70 demo days (Jan 12 - Mar 20)

## 1. Setup

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE DATABASE PDM_DEMO').collect()
session.sql('USE SCHEMA ANALYTICS').collect()
session.sql('USE WAREHOUSE PDM_DEMO_WH').collect()
print('Session ready')

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from snowflake.ml.registry import Registry

np.random.seed(42)
print('Libraries loaded')

## 2. Load Models from Registry & Baselines

In [ ]:
meta_rows = session.sql('SELECT KEY, VALUE FROM PDM_DEMO.ML.MODEL_METADATA').collect()
meta = {r['KEY']: r['VALUE'] for r in meta_rows}
baselines = json.loads(meta['BASELINES'])
model_version = meta['MODEL_VERSION']
vib_mean = baselines['vib_mean']
vib_std = baselines['vib_std']
temp_mean = baselines['temp_mean']
temp_std = baselines['temp_std']
le_classes = np.array(baselines['classes'])

print(f'Model version: {model_version}')
print(f'Normal baselines: VIB={vib_mean:.2f}+/-{vib_std:.2f}, TEMP={temp_mean:.2f}+/-{temp_std:.2f}')
print(f'Classes: {list(le_classes)}')

In [ ]:
registry = Registry(session=session, database_name='PDM_DEMO', schema_name='ML')

print('Models in registry:')
for m in registry.models():
    print(f'  {m.name}')
    for v in m.versions():
        print(f'    {v.version_name}')

clf = registry.get_model('FAILURE_CLASSIFIER').version(model_version).load(force=True)
reg = registry.get_model('RUL_REGRESSOR').version(model_version).load(force=True)
prob_model = registry.get_model('PROBABILITY_CALIBRATOR').version(model_version).load(force=True)

print(f'\nLoaded FAILURE_CLASSIFIER {model_version}: {type(clf).__name__}')
print(f'Loaded RUL_REGRESSOR {model_version}: {type(reg).__name__}')
print(f'Loaded PROBABILITY_CALIBRATOR {model_version}: {type(prob_model).__name__}')

## 3. Load Feature Store Data (Demo Dates)

In [ ]:
DEMO_START = '2026-01-12'
DEMO_END = '2026-03-20 23:59:59'

FEATURE_COLS = [
    'VIBRATION_MEAN_24H', 'VIBRATION_STD_24H', 'VIBRATION_MAX_24H', 'VIBRATION_TREND',
    'TEMPERATURE_MEAN_24H', 'TEMPERATURE_STD_24H', 'TEMPERATURE_MAX_24H', 'TEMPERATURE_TREND',
    'PRESSURE_MEAN_24H', 'PRESSURE_STD_24H', 'FLOW_RATE_MEAN_24H',
    'RPM_MEAN_24H', 'RPM_STD_24H', 'POWER_DRAW_MEAN_24H',
    'DAYS_SINCE_MAINTENANCE', 'MAINTENANCE_COUNT_90D', 'OPERATING_HOURS',
    'DIFF_PRESSURE_MEAN_24H', 'SEAL_TEMP_MEAN_24H',
    'DISCHARGE_TEMP_MEAN_24H', 'COMPRESSION_RATIO_MEAN', 'OIL_PRESSURE_MEAN_24H',
    'IS_PUMP',
    'VIB_TEMP_INTERACTION', 'POWER_EFFICIENCY', 'PRESSURE_VARIABILITY',
    'VIB_DEVIATION', 'TEMP_DEVIATION', 'MAINT_RECENCY_SCORE',
    'VIB_SEVERITY', 'TEMP_SEVERITY', 'DEGRADATION_INTENSITY',
]

pdf = session.sql(f"""
    SELECT f.*, a.ASSET_TYPE as ATYPE
    FROM PDM_DEMO.ANALYTICS.FEATURE_STORE f
    JOIN PDM_DEMO.RAW.ASSETS a ON f.ASSET_ID = a.ASSET_ID
    WHERE f.AS_OF_TS >= '{DEMO_START}' AND f.AS_OF_TS <= '{DEMO_END}'
    ORDER BY f.ASSET_ID, f.AS_OF_TS
""").to_pandas()
pdf = pdf.sort_values(['ASSET_ID', 'AS_OF_TS']).drop_duplicates(['ASSET_ID', 'AS_OF_TS'], keep='last')

if 'ASSET_TYPE' not in pdf.columns and 'ATYPE' in pdf.columns:
    pdf['ASSET_TYPE'] = pdf['ATYPE']

print(f'Loaded {len(pdf):,} rows for {pdf["ASSET_ID"].nunique()} assets')
print(f'Date range: {pdf["AS_OF_TS"].min()} → {pdf["AS_OF_TS"].max()}')

## 4. Engineer Features (must match training)

In [ ]:
pdf['IS_PUMP'] = (pdf['ASSET_TYPE'] == 'PUMP').astype(int)
pdf['VIB_TEMP_INTERACTION'] = pdf['VIBRATION_MEAN_24H'] * pdf['TEMPERATURE_MEAN_24H'] / 1000
pdf['POWER_EFFICIENCY'] = pdf['POWER_DRAW_MEAN_24H'] / (pdf['FLOW_RATE_MEAN_24H'] + 1)
pdf['PRESSURE_VARIABILITY'] = pdf['PRESSURE_STD_24H'] / (pdf['PRESSURE_MEAN_24H'] + 1)
pdf['VIB_DEVIATION'] = pdf['VIBRATION_MAX_24H'] - pdf['VIBRATION_MEAN_24H']
pdf['TEMP_DEVIATION'] = pdf['TEMPERATURE_MAX_24H'] - pdf['TEMPERATURE_MEAN_24H']
pdf['MAINT_RECENCY_SCORE'] = 1 / (pdf['DAYS_SINCE_MAINTENANCE'] + 1)

pdf['VIB_SEVERITY'] = (pdf['VIBRATION_MEAN_24H'] - vib_mean) / vib_std
pdf['TEMP_SEVERITY'] = (pdf['TEMPERATURE_MEAN_24H'] - temp_mean) / temp_std
pdf['DEGRADATION_INTENSITY'] = np.sqrt(pdf['VIB_SEVERITY'].clip(lower=0)**2 + pdf['TEMP_SEVERITY'].clip(lower=0)**2)

for col in FEATURE_COLS:
    if col in pdf.columns:
        pdf[col] = pdf[col].fillna(0)

print(f'Engineered {len(FEATURE_COLS)} features')

## 5. Score with ML Models

In [ ]:
X_score = pdf[FEATURE_COLS].values
cls_preds = clf.predict(X_score)
rul_preds = reg.predict(X_score)
cls_probas = prob_model.predict_proba(X_score)

predictions = []
for i in range(len(pdf)):
    row = pdf.iloc[i]
    probas = {str(le_classes[j]): round(float(cls_probas[i][j]), 4) for j in range(len(le_classes))}
    label = str(le_classes[np.argmax(cls_probas[i])])  # Use argmax of calibrated probs
    rul = max(0, round(float(rul_preds[i]), 1))

    if label == 'NORMAL':
        rul = None
        risk = 'HEALTHY'
    elif rul <= 2:
        rul = 0.0
        label = 'OFFLINE'
        risk = 'OFFLINE'
    elif rul <= 7:
        risk = 'CRITICAL'
    elif rul <= 30:
        risk = 'WARNING'
    else:
        risk = 'HEALTHY'

    ts_str = row['AS_OF_TS'].strftime('%Y-%m-%d %H:%M:%S') if hasattr(row['AS_OF_TS'], 'strftime') else str(row['AS_OF_TS'])
    predictions.append({
        'ASSET_ID': int(row['ASSET_ID']),
        'AS_OF_TS': ts_str,
        'PREDICTED_CLASS': label,
        'CLASS_PROBABILITIES': json.dumps(probas),
        'PREDICTED_RUL_DAYS': rul,
        'RISK_LEVEL': risk,
        'MODEL_VERSION': model_version,
        'SCORED_AT': ts_str,
    })

pred_df = pd.DataFrame(predictions)
pred_df = pred_df.sort_values(['ASSET_ID', 'AS_OF_TS']).drop_duplicates(
    ['ASSET_ID', 'AS_OF_TS'], keep='last'
).reset_index(drop=True)

print(f'Scored {len(pred_df):,} predictions')
print(f'\nRisk distribution:')
print(pred_df['RISK_LEVEL'].value_counts().to_string())

In [ ]:
print('At-risk assets at 2026-03-13:')
now_preds = pred_df[(pred_df['AS_OF_TS'].str.contains('2026-03-13')) & (pred_df['RISK_LEVEL'] != 'HEALTHY')]
for _, r in now_preds.sort_values('PREDICTED_RUL_DAYS').iterrows():
    print(f'  Asset {int(r.ASSET_ID):>2}: {r.PREDICTED_CLASS:<15} RUL={r.PREDICTED_RUL_DAYS:>5.1f}d  {r.RISK_LEVEL}')

In [ ]:
a27 = pred_df[pred_df['ASSET_ID'] == 27].copy()
a27['DT'] = pd.to_datetime(a27['AS_OF_TS']).dt.date
a27_daily = a27.groupby('DT').first().reset_index()

print('Asset 27 — Class Probability Progression:')
print(f'{"Date":<12} {"Class":<16} {"RUL":>6} {"Risk":<10} {"P(BEARING_WEAR)":>16} {"P(NORMAL)":>10}')
print('-' * 80)
for _, r in a27_daily.iterrows():
    probs = json.loads(r['CLASS_PROBABILITIES'])
    p_bw = probs.get('BEARING_WEAR', 0)
    p_n = probs.get('NORMAL', 0)
    print(f'{str(r["DT"]):<12} {r["PREDICTED_CLASS"]:<16} {r["PREDICTED_RUL_DAYS"]:>6} {r["RISK_LEVEL"]:<10} {p_bw:>16.4f} {p_n:>10.4f}')

## 6. Degrading Assets — Full Trajectory

In [ ]:
degrading_ids = [5, 12, 18, 22, 27, 34, 35, 39, 41, 48]

print(f'{"Asset":>5}  {"Day 0 (03-06)":<22}  {"Day 7 (03-13)":<22}  {"Day 14 (03-20)":<22}')
print('=' * 80)
for aid in degrading_ids:
    ap = pred_df[pred_df['ASSET_ID'] == aid].sort_values('AS_OF_TS')
    if len(ap) == 0:
        continue
    first = ap.iloc[0]
    mid_rows = ap[ap['AS_OF_TS'].str.startswith('2026-03-13')]
    mid = mid_rows.iloc[0] if len(mid_rows) > 0 else first
    last = ap.iloc[-1]
    print(
        f'{aid:>5}  {first["PREDICTED_CLASS"]:<12}({first["PREDICTED_RUL_DAYS"]})  '
        f'{mid["PREDICTED_CLASS"]:<12}({mid["PREDICTED_RUL_DAYS"]})  '
        f'{last["PREDICTED_CLASS"]:<12}({last["PREDICTED_RUL_DAYS"]})'
    )

## 7. Materialize to PREDICTIONS Table

In [ ]:
session.sql("""CREATE TABLE IF NOT EXISTS PDM_DEMO.ANALYTICS.PREDICTIONS (
    ASSET_ID INT, AS_OF_TS TIMESTAMP_NTZ, PREDICTED_CLASS VARCHAR(30),
    CLASS_PROBABILITIES VARIANT, PREDICTED_RUL_DAYS FLOAT,
    RISK_LEVEL VARCHAR(20), MODEL_VERSION VARCHAR(20), SCORED_AT TIMESTAMP_NTZ
)""").collect()

sdf = session.create_dataframe(pred_df)
sdf.write.mode('overwrite').save_as_table('PDM_DEMO.ANALYTICS.PREDICTIONS')

count = session.sql('SELECT COUNT(*) as N FROM PDM_DEMO.ANALYTICS.PREDICTIONS').collect()[0]['N']
print(f'Materialized {count:,} predictions to PDM_DEMO.ANALYTICS.PREDICTIONS')

verify = session.sql("""
    SELECT RISK_LEVEL, COUNT(*) as N, COUNT(DISTINCT ASSET_ID) as ASSETS
    FROM PDM_DEMO.ANALYTICS.PREDICTIONS
    WHERE AS_OF_TS LIKE '2026-03-13%'
    GROUP BY RISK_LEVEL ORDER BY 1
""").to_pandas()
print(f'\nRisk distribution at 2026-03-13:')
print(verify.to_string(index=False))

print('\nDone — predictions ready for application layer')